In [1]:
import torch
import opt_einsum

In [3]:
def two_norm_power_iteration(B, norm2_0, x_0, tol, max_iterations, dtype, device):
    """
    Estimate the 2-norm (largest singular value) of a batch of matrices 
    using power iteration on B^* B implicitly.

    Args:
        B : torch.Tensor of shape (N, M, K)
            Batch of N matrices (not necessarily square).
        norm2_0 : float
            Initial guess for 2-norm.
        x_0 : torch.Tensor of shape (K,)
            Initial guess for right singular vector (normalized). Tiled across batch.
        tol : float
            Convergence tolerance (relative change in norm).
        max_iterations : int
            Maximum number of iterations.
        dtype : torch.dtype
            Tensor data type (e.g., torch.float32 or torch.float64).
        device : torch.device
            Device (e.g., torch.device('cuda') or 'cpu').

    Returns:
        norm2 : torch.Tensor of shape (N,)
            Estimated 2-norms for each matrix.
        v : torch.Tensor of shape (K,)
            Approximate right singular vector for the first matrix.
    """
    N, M, K = B.shape
    x = x_0 / torch.clamp(torch.linalg.norm(x_0), min=1e-7)
    x = x.repeat(N, 1)  # Shape: (N, K)
    norm2 = norm2_0 * torch.ones(N, dtype=dtype, device=device)

    for i in range(max_iterations):
        # 1) y = B x          shape (N, M)
        y = torch.einsum('nij,nj->ni', B, x)

        # 2) norm of y (for each matrix in batch)
        norm_y = torch.linalg.norm(y, dim=1)

        # 3) z = B^* y         shape (N, K)
        z = torch.einsum('nij,ni->nj', B, y)

        # Normalize z to get next x
        z_norm = torch.linalg.norm(z, dim=1)
        z_norm = torch.clamp(z_norm, min=1e-7)
        x_new = z / z_norm[:, None]

        # Update norm2 estimate
        norm2_new = norm_y

        # Relative difference for convergence check
        reldiff = torch.abs((norm2_new - norm2) / torch.clamp(norm2, min=1e-7))

        # Update norm2 and x only where not converged
        not_converged = reldiff > tol
        norm2[not_converged] = norm2_new[not_converged]
        x[not_converged] = x_new[not_converged]

        if torch.sum(not_converged) == 0:
            break

    print(f"Total iterations: {i+1}")

    if i == max_iterations - 1:
        print('Power iteration did not fully converge within max_iterations. Number of non-converged matrices:', torch.sum(reldiff > tol).item())
        # Fall back to exact computation for non-converged matrices
        norm2[not_converged] = torch.linalg.norm(B[not_converged], ord=2, dim=(1, 2))

    # Return the right singular vector for the first matrix
    v = x[0] / torch.linalg.norm(x[0])

    return norm2, v


In [4]:
B = torch.randn(10,100,100)

In [23]:
two_norm_power_iteration(B=B, norm2_0=1, x_0=torch.randn(100), tol=1e-4, max_iterations=1000, dtype=torch.float32, device='cpu')

Total iterations: 85


(tensor([19.4689, 19.4977, 19.3183, 20.1922, 20.2783, 19.3772, 19.7938, 19.3443,
         19.3229, 19.2519]),
 tensor([ 0.0671,  0.0485, -0.2310, -0.0616, -0.0748,  0.0842, -0.1312, -0.1040,
          0.0972,  0.0887,  0.0288,  0.1750, -0.0112, -0.0019,  0.0811,  0.0274,
         -0.0277,  0.1484, -0.0070,  0.0741,  0.0588, -0.1544,  0.0563,  0.0399,
          0.0083, -0.0755,  0.1030,  0.0463, -0.0743, -0.0269,  0.0090, -0.1676,
          0.1455,  0.1030, -0.0654, -0.0003, -0.0858, -0.0334,  0.0124, -0.0729,
          0.0132,  0.0716, -0.1353, -0.1519,  0.1063,  0.2002,  0.0519,  0.0273,
         -0.0885, -0.2412, -0.0632, -0.1463,  0.0752, -0.0528, -0.1480, -0.1168,
          0.0225, -0.1899, -0.0942, -0.0779, -0.0633, -0.0174,  0.1202,  0.1873,
         -0.0308,  0.0210, -0.0782, -0.0287,  0.0495, -0.1855, -0.1736, -0.0242,
          0.1441,  0.0770,  0.0867, -0.0020,  0.1510,  0.0720,  0.0190,  0.0792,
          0.0636,  0.0118,  0.0270, -0.0871, -0.0764, -0.0306,  0.0171, -0.1649,

In [24]:
torch.sqrt(torch.linalg.eigvals(B.transpose(-2,-1) @ B).abs().max(dim=-1).values)

tensor([19.4800, 19.5303, 19.3359, 20.1955, 20.2848, 19.4098, 19.8024, 19.3654,
        19.3634, 19.2980])